# 🎬 Content Recommendation System – ML Exploration Notebook

This notebook walks through the complete machine learning pipeline:
1. Data loading and preprocessing
2. Content-Based Filtering
3. Collaborative Filtering  
4. Hybrid System
5. Evaluation Metrics
6. Visualizations


In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import plotly.express as px
import plotly.graph_objects as go
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

plt.style.use('dark_background')
plt.rcParams.update({'figure.facecolor': '#0f1219', 'axes.facecolor': '#141822', 'font.family': 'monospace'})
print('✅ Libraries loaded')

## 1. Create Sample Dataset
We simulate the MovieLens structure with sample data.

In [ ]:
# ── Sample movies (simulates MovieLens u.item) ───────────────
movies = pd.DataFrame([
    {'id':1, 'title':'The Dark Knight',        'genre':'Action,Crime,Drama',     'year':2008},
    {'id':2, 'title':'Inception',              'genre':'Action,Sci-Fi',          'year':2010},
    {'id':3, 'title':'Interstellar',           'genre':'Adventure,Drama,Sci-Fi', 'year':2014},
    {'id':4, 'title':'The Matrix',             'genre':'Action,Sci-Fi',          'year':1999},
    {'id':5, 'title':'Pulp Fiction',           'genre':'Crime,Drama',            'year':1994},
    {'id':6, 'title':'The Shawshank Redemption','genre':'Drama',                 'year':1994},
    {'id':7, 'title':'Forrest Gump',           'genre':'Drama,Romance',          'year':1994},
    {'id':8, 'title':'The Godfather',          'genre':'Crime,Drama',            'year':1972},
    {'id':9, 'title':'Avengers: Endgame',      'genre':'Action,Adventure,Sci-Fi','year':2019},
    {'id':10,'title':'Parasite',               'genre':'Comedy,Drama,Thriller',  'year':2019},
    {'id':11,'title':'Get Out',                'genre':'Horror,Thriller',        'year':2017},
    {'id':12,'title':'La La Land',             'genre':'Comedy,Drama,Music',     'year':2016},
])

# ── Sample ratings (simulates MovieLens u.data) ──────────────
np.random.seed(42)
ratings_data = []
for user_id in range(1, 8):
    for movie_id in np.random.choice(range(1, 13), size=np.random.randint(4, 10), replace=False):
        ratings_data.append({'user_id': user_id, 'content_id': int(movie_id),
                              'rating': np.random.choice([1,2,3,4,5], p=[0.05,0.1,0.2,0.4,0.25])})

ratings = pd.DataFrame(ratings_data)
print(f'Movies: {len(movies)} | Ratings: {len(ratings)} | Users: {ratings["user_id"].nunique()}')
ratings.head()

## 2. Content-Based Filtering
Build TF-IDF vectors from genre strings and compute cosine similarity.

In [ ]:
# ── Step 1: Build feature 'soup' ─────────────────────────────
movies['soup'] = movies['genre'].str.replace(',', ' ') + ' ' + movies['year'].astype(str)
print('Sample soup:', movies['soup'].iloc[0])

# ── Step 2: TF-IDF Vectorization ─────────────────────────────
tfidf    = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['soup'])
print(f'TF-IDF matrix shape: {tfidf_matrix.shape}  (movies × unique_terms)')

# ── Step 3: Cosine Similarity ─────────────────────────────────
cos_sim = cosine_similarity(tfidf_matrix)
print(f'Similarity matrix shape: {cos_sim.shape}  (movies × movies)')

# ── Visualize: Similarity Heatmap ─────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cos_sim, cmap='Reds', vmin=0, vmax=1)
ax.set_xticks(range(len(movies)))
ax.set_yticks(range(len(movies)))
ax.set_xticklabels([t[:15] for t in movies['title']], rotation=45, ha='right', fontsize=8)
ax.set_yticklabels([t[:15] for t in movies['title']], fontsize=8)
ax.set_title('Content-Based Cosine Similarity Matrix', color='white', pad=12)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('../docs/similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Function: Get top-N similar movies ───────────────────────
def get_similar_movies(movie_title, top_n=5):
    idx        = movies[movies['title'] == movie_title].index[0]
    sim_scores = list(enumerate(cos_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = [(i, s) for i, s in sim_scores if i != idx][:top_n]
    
    results = []
    for i, score in sim_scores:
        results.append({'title': movies.iloc[i]['title'],
                        'genre': movies.iloc[i]['genre'],
                        'similarity': round(score, 4)})
    return pd.DataFrame(results)

print('\n🎬 Movies similar to "Inception":')
print(get_similar_movies('Inception').to_string(index=False))

## 3. Collaborative Filtering
Build user-item matrix and find similar users.

In [ ]:
# ── Build User-Item Matrix ────────────────────────────────────
user_movie_matrix = ratings.pivot_table(
    index='user_id', columns='content_id',
    values='rating', fill_value=0
)
print('User-Item Matrix shape:', user_movie_matrix.shape)
print(f'Sparsity: {(user_movie_matrix == 0).sum().sum() / user_movie_matrix.size * 100:.1f}% zeros\n')
user_movie_matrix

In [ ]:
# ── User Similarity ───────────────────────────────────────────
user_sim = cosine_similarity(user_movie_matrix)
user_sim_df = pd.DataFrame(user_sim,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index)

print('User Similarity Matrix:')
print(user_sim_df.round(3))

# Visualize
fig = px.imshow(user_sim_df, color_continuous_scale='Reds',
                title='User-User Similarity Matrix (Collaborative Filtering)',
                template='plotly_dark')
fig.show()

In [ ]:
# ── Predict Rating: User 1, Movie 5 ──────────────────────────
TARGET_USER   = 1
TARGET_MOVIE  = 5
K_NEIGHBORS   = 3

user_idx = list(user_movie_matrix.index).index(TARGET_USER)
sims     = user_sim[user_idx].copy()
sims[user_idx] = -1  # exclude self

top_k_users = np.argsort(sims)[::-1][:K_NEIGHBORS]
print(f'Top-{K_NEIGHBORS} users similar to User {TARGET_USER}:')
for i in top_k_users:
    uid = user_movie_matrix.index[i]
    print(f'  User {uid}: similarity = {sims[i]:.3f}')

# Weighted average prediction
numerator, denominator = 0, 0
for i in top_k_users:
    uid    = user_movie_matrix.index[i]
    rating = user_movie_matrix.loc[uid, TARGET_MOVIE] if TARGET_MOVIE in user_movie_matrix.columns else 0
    if rating > 0:
        numerator   += sims[i] * rating
        denominator += sims[i]

pred = numerator / denominator if denominator > 0 else 0
print(f'\nPredicted rating for User {TARGET_USER}, Movie {TARGET_MOVIE}: {pred:.2f} ★')

## 4. Rating Distribution Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Rating distribution
ratings['rating'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='#e63946', edgecolor='#1e2536')
axes[0].set_title('Rating Distribution', color='white')
axes[0].set_xlabel('Rating'); axes[0].set_ylabel('Count')

# Ratings per user
ratings.groupby('user_id')['rating'].count().plot(
    kind='bar', ax=axes[1], color='#ffd166', edgecolor='#1e2536')
axes[1].set_title('Ratings per User', color='white')
axes[1].set_xlabel('User ID')

# Genre counts
genre_series = movies['genre'].str.split(',').explode().str.strip()
genre_series.value_counts().plot(
    kind='barh', ax=axes[2], color='#06d6a0', edgecolor='#1e2536')
axes[2].set_title('Genre Distribution', color='white')

for ax in axes:
    ax.tick_params(colors='gray')

plt.tight_layout()
plt.savefig('../docs/eda_charts.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Evaluation Metrics

In [ ]:
from math import sqrt
from sklearn.metrics import mean_squared_error

# Simulate ground truth vs predictions
np.random.seed(7)
actual    = [4, 3, 5, 2, 4, 3, 5, 1, 4]
predicted = [3.8, 3.2, 4.5, 2.1, 3.9, 3.5, 4.8, 1.2, 3.7]

rmse = sqrt(mean_squared_error(actual, predicted))
mae  = np.mean(np.abs(np.array(actual) - np.array(predicted)))

print(f'RMSE : {rmse:.4f}  (lower = better, range 0–4)')
print(f'MAE  : {mae:.4f}  (lower = better, range 0–4)')

# Precision, Recall, F1 @ K
recommended = [1, 3, 5, 7, 9, 2, 4, 6, 8, 10]  # ordered list
relevant    = {1, 3, 9, 5}  # items user actually liked
K = 10

hits      = len([r for r in recommended[:K] if r in relevant])
precision = hits / K
recall    = hits / len(relevant)
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f'\nPrecision@{K}: {precision:.4f}  (fraction of top-{K} that are relevant)')
print(f'Recall@{K}   : {recall:.4f}  (fraction of relevant found in top-{K})')
print(f'F1@{K}       : {f1:.4f}  (harmonic mean of P and R)')

# Bar chart of metrics
metrics  = ['Precision@10', 'Recall@10', 'F1@10', 'Normalised\nRMSE']
values   = [precision, recall, f1, 1 - rmse/4]
colors   = ['#e63946', '#ffd166', '#06d6a0', '#4cc9f0']
fig, ax  = plt.subplots(figsize=(8, 4))
bars     = ax.bar(metrics, values, color=colors, edgecolor='#1e2536', width=0.5)
ax.set_ylim(0, 1.2); ax.set_title('Recommendation Evaluation Metrics', color='white')
ax.set_ylabel('Score'); ax.axhline(1.0, color='white', lw=0.5, ls='--')
for bar, val in zip(bars, values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.03,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('../docs/metrics_chart.png', dpi=150, bbox_inches='tight')
plt.show()